# Getting Started with xndbc

This notebook demonstrates the main features of the `xndbc` package for accessing and analyzing NOAA NDBC buoy data.

In [ ]:
import xndbc
import numpy as np
import matplotlib.pyplot as plt
from helpers import compute_data_coverage, plot_stations

print(f"xndbc version: {xndbc.__version__}")

## 1. List Available Stations

Get metadata for all NDBC buoy stations:

In [ ]:
# Get all buoy stations
stations = xndbc.list_available(mode=None) #takes a while to pull from NDBC database! 
print(f"Total stations: {len(stations.station_id)}")
stations

## 2. Filter Stations by Region

Filter stations to a specific geographic area:

In [ ]:
caribbean = xndbc.list_available(
    mode=None,
    lon_min=-85,
    lon_max=-60,
    lat_min=10,
    lat_max=25,
)
print(f"Caribbean stations: {len(caribbean.station_id)}")

## 3. Visualize Station Locations

Plot station locations on a map:

In [ ]:
# Plot Caribbean stations
fig, ax = plot_stations(caribbean)
ax.set_extent([-86, -59, 9, 26])
ax.set_title("NDBC Stations - Caribbean Region")
plt.show()

## 4. Historical Coverage in the Region

Use the historical index to find standard meteorological files for the selected Caribbean stations, then fetch the available regional stations for one recent year.

In [ ]:
available_stdmet = xndbc.list_available(mode="stdmet").to_dataframe()
caribbean_ids = [str(station_id) for station_id in caribbean.station_id.values]
caribbean_stdmet = available_stdmet[available_stdmet.station_id.isin(caribbean_ids)]
stdmet_year = int(caribbean_stdmet.year.max())
stdmet_stations = sorted(caribbean_stdmet.loc[caribbean_stdmet.year == stdmet_year, "station_id"].unique())

regional_data = xndbc.fetch_data(
    station_ids=stdmet_stations,
    years=stdmet_year,
    sample_rate="D",
)
regional_coverage = compute_data_coverage(regional_data)

regional_coverage

In [ ]:
coverage_variable = "WTMP_coverage" if "WTMP_coverage" in regional_coverage else list(regional_coverage.data_vars)[0]
fig, ax = plot_stations(regional_coverage, variable=coverage_variable)
ax.set_extent([-86, -59, 9, 26])
ax.set_title(f"Caribbean {coverage_variable} in {stdmet_year}")
plt.show()

## 5. Water Temperature Across the Region

In [ ]:
wtmp = regional_data["WTMP"].dropna("station_id", how="all")
station_idx = np.arange(len(wtmp.station_id))

plt.figure(figsize=(12, 5))
plt.pcolormesh(wtmp.time, station_idx, wtmp, cmap="viridis", shading="auto")
plt.yticks(station_idx + 0.5, [str(station).upper() for station in wtmp.station_id.values])
plt.xlabel("Time")
plt.ylabel("Station ID")
plt.colorbar(label="WTMP")
plt.title(f"Caribbean Water Temperature in {stdmet_year}")
plt.show()

## 6. Historical Availability

In [ ]:
caribbean_stdmet.head()

## 7. Realtime Data

Realtime feeds use the same `fetch_data` API and return an `xarray.Dataset`.

In [ ]:
realtime_station = str(regional_data.station_id.isel(station_id=0).item())

realtime = xndbc.fetch_data(
    realtime_station,
    data_type="realtime",
    sample_rate="H",
)

realtime

## 8. ADCP Data in the Region

ADCP files are reshaped onto a `depth_bin` dimension.

In [ ]:
available_adcp = xndbc.list_available(mode="adcp").to_dataframe()
caribbean_adcp = available_adcp[available_adcp.station_id.isin(caribbean_ids)]
adcp_station = str(caribbean_adcp.station_id.iloc[0])
adcp_year = int(caribbean_adcp.year.iloc[0])

adcp = xndbc.fetch_data(
    adcp_station,
    years=adcp_year,
    sample_rate="D",
    mode="adcp",
)

adcp

In [ ]:
adcp_variable = "SPD" if "SPD" in adcp else list(adcp.data_vars)[0]
adcp[adcp_variable].isel(station_id=0).plot(figsize=(10, 4))
plt.title(f"{adcp_variable} at {adcp_station.upper()} in {adcp_year}")
plt.show()

## 9. Spectral Wave Data in the Region

Spectral wave files are reshaped onto a `frequency` dimension.

In [ ]:
available_swden = xndbc.list_available(mode="swden").to_dataframe()
caribbean_swden = available_swden[available_swden.station_id.isin(caribbean_ids)]
spectral_station = str(caribbean_swden.station_id.iloc[0])
spectral_year = int(caribbean_swden.year.iloc[0])

spectral = xndbc.fetch_data(
    spectral_station,
    years=spectral_year,
    sample_rate="D",
    mode="swden",
)

spectral

In [ ]:
spectral_variable = "SWDEN" if "SWDEN" in spectral else list(spectral.data_vars)[0]
spectral[spectral_variable].isel(station_id=0).plot(figsize=(10, 4))
plt.title(f"{spectral_variable} at {spectral_station.upper()} in {spectral_year}")
plt.show()